# Multi-turn policy comparison (calming vs. provocative)

Runs short 5-turn conversations across all seven emotions with three deterministic policies (calming, provocative, always-validate baseline). Saves per-turn logs, summaries, and heatmaps under `results/multiturn_runs/`. Set `OPENAI_API_KEY` before running; GPU is optional but recommended.

In [ ]:
import os
from pathlib import Path
import pandas as pd
from dynamic_conversation import (
    EmotionFlowAnalyzer,
    MultiTurnRollout,
    calming_policy,
    provocative_policy,
    always_validate_policy,
)

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# Configuration
emotions = EmotionFlowAnalyzer.EMOTIONS
policies = {
    "calming": calming_policy,
    "provocative": provocative_policy,
    "validate": always_validate_policy,
}
runs_per_emotion = 3  # keep small to finish within ~20–30 minutes
turns = 5
style_modifier = "concise and emotionally attuned"
out_dir = Path("results/multiturn_runs")
out_dir.mkdir(parents=True, exist_ok=True)

sim = MultiTurnRollout(
    use_gpu=True,
    turns=turns,
    default_style=style_modifier,
)
print(f"Device set to GPU? {sim.emotion_analyzer.classifier.device != -1}")

In [ ]:
# Run all policies
per_policy = {}
for name, fn in policies.items():
    csv_path = out_dir / f"{name}_5turn.csv"
    plot_path = out_dir / f"{name}_5turn_heatmap.png"
    per_turn_df, summary_df = sim.run_policy_batch(
        policy_name=name,
        policy_fn=fn,
        emotions=emotions,
        runs_per_emotion=runs_per_emotion,
        turns=turns,
        style_modifier=style_modifier,
        use_llm_seed=True,
        use_esconv_seed=False,
        save_csv=csv_path,
        save_plot=plot_path,
    )
    per_policy[name] = (per_turn_df, summary_df)
    display(summary_df.head())
print("✓ Completed all policy runs")

In [ ]:
# Simple comparisons: final emotion distribution and trajectory means
rows = []
for name, (turn_df, summary_df) in per_policy.items():
    if turn_df.empty:
        continue
    last_turn = turn_df['turn'].max()
    finals = turn_df[turn_df['turn'] == last_turn]
    dist = finals['detected_emotion'].value_counts(normalize=True)
    traj_mean = summary_df['trajectory_to_intended'].mean() if not summary_df.empty else 0.0
    rows.append({
        'policy': name,
        'trajectory_to_intended_mean': traj_mean,
        'final_top_emotion': dist.idxmax() if not dist.empty else 'n/a',
        'final_top_prop': dist.max() if not dist.empty else 0.0,
    })
comparison_df = pd.DataFrame(rows).sort_values(by='trajectory_to_intended_mean', ascending=False)
display(comparison_df)

## Notes
- Outputs: per-turn logs at `results/multiturn_runs/<policy>_5turn.csv`, summaries at `.summary.csv`, heatmaps at `_5turn_heatmap.png`.
- `trajectory_to_intended` scores weight later turns; higher means a stronger pull toward the starting emotion.
- Adjust `runs_per_emotion` or `turns` if runtime is too high; keep seeds consistent across policies by controlling randomness in future iterations.